# Segment 3: Top-K Dataset Image Extraction

**Goal**: For each channel in a given InceptionV1 layer, find the dataset
images that *maximally activate* that channel and save both full images
and spatially-cropped patches to disk.

## Pipeline Overview

| Pass | Description |
|------|-------------|
| **Pass 1 — Scan** | Stream ImageNet-1k, compute activations, maintain per-channel min-heaps of top-k scoring images (keys only). |
| **Pass 2 — Save** | Re-stream the dataset, match keys from Pass 1, save FULL + CROP images to disk. |

## Memory Budget (8 GB GPU)

| Component | Estimate |
|-----------|----------|
| InceptionV1 weights | ~25 MB |
| Batch (64 × 3 × 224 × 224 FP16) | ~18 MB |
| mixed4e activations (64 × 832 × 14 × 14 FP16) | ~20 MB |
| PyTorch overhead | ~500 MB |
| **Total** | **~565 MB** ✅ |

## Activation Scoring

$$\text{score}_{c}(x) = \max_{i,j} \; A^c_{i,j}(x)$$

Where $A^c_{i,j}(x)$ is the activation of channel $c$ at spatial location
$(i, j)$ for input image $x$.

## 1. Environment Setup

Configure the device, authenticate with HuggingFace, and verify GPU memory.

In [1]:
"""Environment setup: device detection, HF auth, GPU memory check."""
import os
import gc
import re
import heapq
import pickle as pkl
from pathlib import Path

import torch
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

# HuggingFace token (replace with your own)
import os
HF_TOKEN = os.environ.get('HF_TOKEN') # ⚠️ SET HF_TOKEN ENV VAR OR REPLACE THIS STRING
os.environ["HF_TOKEN"] = HF_TOKEN
print(f"HF_TOKEN loaded (length={len(HF_TOKEN)})")

# Device setup with detailed GPU info
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

if device.type == "cuda":
    gpu_name = torch.cuda.get_device_name(0)
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    free_mem = torch.cuda.mem_get_info(0)[0] / 1024**3
    print(f"GPU: {gpu_name} ({total_mem:.1f} GB total, {free_mem:.1f} GB free)")
else:
    print("⚠ No CUDA GPU detected — pipeline will be very slow on CPU.")

HF_TOKEN loaded (length=37)
Device: cuda:0
GPU: NVIDIA GeForce RTX 2070 (8.0 GB total, 7.0 GB free)


## 2. Configuration

All tunable parameters in one place.

| Parameter | Description | Default |
|-----------|-------------|---------|
| `LAYER_NAME` | Target InceptionV1 layer | `mixed5a` |
| `CHANNEL_START/END` | Channel range to scan | `0–832` |
| `BATCH_SIZE` | Images per batch (64 for 8 GB GPU) | `192` |
| `TOPK` | Top images to keep per channel | `10` |
| `SAVE_EVERY` | Checkpoint frequency (batches) | `500` |
| `CROP_FRAC` | Crop window as fraction of image | `0.25` |

In [2]:
"""Pipeline configuration — all tunable parameters in one place."""

# Project root (parent of notebooks/)
PROJECT_ROOT = Path.cwd().parent

CFG = {
    "LAYER_NAME": "mixed5a",
    "CHANNEL_START": 0,
    "CHANNEL_END": 832,
    "TOPK": 10,
    "HEAP_BATCH": 20,
    "REDUCTION": "max",
    "BATCH_SIZE": 192,           # Halved from 128 for 8 GB GPU safety
    "CROP_FRAC": 0.25,
    "SAVE_EVERY": 500,          # Checkpoint every 500 batches
    "OUT_ROOT": PROJECT_ROOT / "notebooks" / "results" / "dataset_images",
    "CKPT_DIR": PROJECT_ROOT / "notebooks" / "results" / "checkpoints",
}

# Eagerly create output directories
CFG["OUT_ROOT"].mkdir(parents=True, exist_ok=True)
CFG["CKPT_DIR"].mkdir(parents=True, exist_ok=True)
print(f"Output:      {CFG['OUT_ROOT']}")
print(f"Checkpoints: {CFG['CKPT_DIR']}")

Output:      c:\Users\cataluna84\Documents\Workspace\VisionInterpretability\notebooks\results\dataset_images
Checkpoints: c:\Users\cataluna84\Documents\Workspace\VisionInterpretability\notebooks\results\checkpoints


## 3. Model Loading

Load InceptionV1 (GoogLeNet variant from Lucent) pretrained on ImageNet.
The model is ~25 MB — negligible relative to the 8 GB budget.

In [3]:
"""Load the pretrained InceptionV1 model."""
from lucent.modelzoo import inceptionv1

model = inceptionv1(pretrained=True).to(device).eval()

# Verify the target layer exists
layer_name = CFG["LAYER_NAME"]
available_layers = dict(model.named_modules()).keys()
if layer_name not in available_layers:
    raise ValueError(
        f"Layer '{layer_name}' not found. "
        f"Available: {sorted(available_layers)}"
    )
print(f"Model loaded. Target layer: {layer_name}")

if device.type == "cuda":
    mem_used = torch.cuda.memory_allocated(0) / 1024**2
    print(f"GPU memory after model load: {mem_used:.0f} MB")

Model loaded. Target layer: mixed5a
GPU memory after model load: 27 MB


## 4. Dataset: ImageNet-1k (Local Download)

Downloads the full ImageNet-1k WDS shards (~144 GB) to
`data/imagenet-1k-wds/` using `huggingface_hub`. The download
**automatically resumes** if interrupted.

### Performance

| Approach | Throughput |
|----------|------------|
| HTTPS streaming (urllib) | Slow (buffering + GIL) |
| **Local shards + num_workers=2** | **Fast (native I/O)** |

With local files, PyTorch's built-in DataLoader workers handle
parallel loading efficiently -- no custom threading needed.

> **Note**: WebDataset iterators are **single-use**. The `make_dataloader()`
> factory function creates a fresh DataLoader for each pipeline pass.


In [4]:
"""ImageNet-1k local dataset via WebDataset."""
import glob
import sys

import webdataset as wds
from torch.utils.data import DataLoader
from torchvision import transforms
from huggingface_hub import snapshot_download

# ── Windows fix: register drive-letter schemes in WebDataset ──
# urlparse("c:/path") parses "c" as a URL scheme, causing gopen
# to fail with "no gopen handler defined". Register all drive
# letters to use gopen_file so native Windows paths work.
if sys.platform == "win32":
    from webdataset.gopen import gopen_schemes, gopen_file
    for _letter in "abcdefghijklmnopqrstuvwxyz":
        gopen_schemes[_letter] = gopen_file
        gopen_schemes[_letter.upper()] = gopen_file
    # Also patch islocal() and StreamingOpen to
    # recognise single-letter (drive) schemes as local.
    import webdataset.cache as _wds_cache
    _orig_islocal = _wds_cache.islocal
    def _patched_islocal(url):
        from urllib.parse import urlparse as _up
        s = _up(url).scheme
        if len(s) == 1 and s.isalpha():
            return True
        return _orig_islocal(url)
    _wds_cache.islocal = _patched_islocal

    # Patch StreamingOpen.__call__ to open drive-letter
    # paths directly with open() instead of gopen().
    _OrigStreamingOpen = _wds_cache.StreamingOpen
    _orig_call = _OrigStreamingOpen.__call__
    def _patched_streaming_call(self, urls):
        for url in urls:
            if isinstance(url, dict):
                url = url["url"]
            try:
                stream = open(url, "rb")
                yield dict(url=url, stream=stream, local_path=url)
            except Exception as exn:
                if self.handler(exn):
                    continue
                else:
                    break
    _OrigStreamingOpen.__call__ = _patched_streaming_call

# ── Step 1: Download dataset locally (resumes if interrupted) ──
DATA_DIR = PROJECT_ROOT / "data" / "imagenet-1k-wds"
DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Dataset directory: {DATA_DIR}")
print("Downloading ImageNet-1k WDS shards (resumes if interrupted)...")
print("This is ~144 GB. First run will take a while.")

snapshot_download(
    repo_id="timm/imagenet-1k-wds",
    repo_type="dataset",
    local_dir=str(DATA_DIR),
    token=HF_TOKEN,
    allow_patterns="imagenet1k-train-*.tar",
)

# Find all local shard files
_LOCAL_SHARDS = sorted(glob.glob(str(DATA_DIR / "imagenet1k-train-*.tar")))
print(f"Download complete: {len(_LOCAL_SHARDS)} shards found locally.")

# ── Step 2: Preprocessing ──
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])


def _is_not_none(x: tuple | None) -> bool:
    """Filter predicate: returns True if sample is not None."""
    return x is not None


def to_tensor(sample: dict) -> tuple | None:
    """Convert a WDS sample to (tensor, key), or None if no image.

    Args:
        sample: WebDataset sample dictionary.

    Returns:
        Tuple of (preprocessed_tensor, sample_key), or None.
    """
    img = None
    for ext in ("jpg", "jpeg", "JPEG", "png"):
        img = sample.get(ext)
        if img is not None:
            break
    if img is None:
        return None
    key = sample.get("__key__", "")
    return preprocess(img), key


# ── Step 3: DataLoader factory ──
def make_dataloader(batch_size: int = CFG["BATCH_SIZE"]) -> DataLoader:
    """Create a DataLoader from local shards.

    Uses num_workers=0 on Windows (spawn context cannot pickle
    notebook globals) and num_workers=2 on Linux/macOS.

    Args:
        batch_size: Number of images per batch.

    Returns:
        A new DataLoader ready for iteration.
    """
    dataset = (
        wds.WebDataset(
            HEALTHY_SHARDS
            if "HEALTHY_SHARDS" in globals()
            else _LOCAL_SHARDS,
            shardshuffle=False,
            handler=wds.handlers.warn_and_continue,
        )
        .decode("pil", handler=wds.handlers.warn_and_continue)
        .map(to_tensor, handler=wds.handlers.warn_and_continue)
        .select(_is_not_none)
    )
    return DataLoader(
        dataset,
        batch_size=batch_size,
        num_workers=0 if sys.platform == "win32" else 2,
        pin_memory=(device.type == "cuda"),
        persistent_workers=(sys.platform != "win32"),
    )


print(f"DataLoader ready: {len(_LOCAL_SHARDS)} local shards, "
      f"batch_size={CFG['BATCH_SIZE']}, num_workers=2")


Dataset directory: c:\Users\cataluna84\Documents\Workspace\VisionInterpretability\data\imagenet-1k-wds
This is ~144 GB. First run will take a while.


Fetching ... files: 0it [00:00, ?it/s]

Download complete: 1024 shards found locally.
DataLoader ready: 1024 local shards, batch_size=192, num_workers=2


## 4b. Shard Integrity Verification

Before running the pipeline, we perform a **4-layer integrity check**
on every downloaded shard. This is essential because the ~144 GB
download can be interrupted mid-file, leaving truncated or corrupted
`.tar` archives that would cause silent errors during Pass 1/2.

### Verification Layers

| Layer | Check | What it catches |
|-------|-------|------------------|
| **1. Presence** | All 1024 shard numbers exist on disk | Missing shards (not yet downloaded) |
| **2. Size match** | Local file size == HuggingFace LFS expected size | Truncated / partial downloads |
| **3. Tar integrity** | `tarfile.open()` + iterate all members | Structurally corrupt archives |
| **4. Decode spot-check** | Decode first JPEG from every 50th shard | Corrupt image data inside valid tars |

The output is a `HEALTHY_SHARDS` list containing **only verified**
shard paths. The `make_dataloader()` factory uses this list, so the
pipeline never touches a bad file.

> **Note**: This cell is **re-runnable**. As more shards finish
> downloading, re-run to pick them up and extend the healthy set.


In [5]:
"""4-layer shard integrity verification.

Checks presence, file size, tar structure, and sample JPEG decode
for every local shard. Produces a HEALTHY_SHARDS list for the
downstream pipeline.
"""
import io


import json as _json
import tarfile
from urllib.request import urlopen, Request

# ── Configuration ──
EXPECTED_SHARD_COUNT = 1024
SPOT_CHECK_EVERY = 50  # Decode-check every Nth shard
HF_API_URL = (
    "https://huggingface.co/api/datasets/"
    "timm/imagenet-1k-wds/tree/main"
)


# ── Layer 1: Presence Check ──
# Build the set of expected shard filenames (0000 to 1023)
expected_names = {
    f"imagenet1k-train-{i:04d}.tar"
    for i in range(EXPECTED_SHARD_COUNT)
}

# Map each present shard filename to its full path
local_shard_map = {
    Path(p).name: p for p in _LOCAL_SHARDS
}
present_names = set(local_shard_map.keys())

# Identify missing and unexpected files
missing_names = sorted(expected_names - present_names)
unexpected_names = sorted(present_names - expected_names)

print("=" * 60)
print("LAYER 1: Presence Check")
print("=" * 60)
print(f"  Expected:   {EXPECTED_SHARD_COUNT}")
print(f"  Present:    {len(present_names)}")
print(f"  Missing:    {len(missing_names)}")
if missing_names:
    # Show first 10 missing shards for brevity
    preview = missing_names[:10]
    suffix = (
        f" ... and {len(missing_names) - 10} more"
        if len(missing_names) > 10
        else ""
    )
    print(f"  Examples:   {preview}{suffix}")
if unexpected_names:
    print(f"  Unexpected: {unexpected_names}")


# ── Layer 2: Size Match against HuggingFace LFS Metadata ──
print()
print("=" * 60)
print("LAYER 2: Size Match (HuggingFace LFS)")
print("=" * 60)

# Fetch expected sizes from HuggingFace API
print("  Fetching expected sizes from HuggingFace API...")
expected_sizes = {}  # filename -> expected bytes
try:
    # The tree API is paginated; fetch all pages
    cursor = None
    while True:
        url = HF_API_URL
        if cursor:
            url += f"?cursor={cursor}"
        req = Request(url)
        req.add_header("Authorization", f"Bearer {HF_TOKEN}")
        with urlopen(req, timeout=30) as resp:
            page = _json.loads(resp.read())
        for entry in page:
            name = entry.get("path", "")
            if (
                name.startswith("imagenet1k-train-")
                and name.endswith(".tar")
            ):
                # Use LFS size (authoritative) if available,
                # otherwise fall back to top-level size
                lfs = entry.get("lfs", {})
                size = lfs.get("size", entry.get("size", 0))
                expected_sizes[name] = int(size)

        # Check for next page or break
        if len(page) < 50:
            break
        # HuggingFace uses the last item's path as cursor
        cursor = page[-1].get("path", "")

    print(f"  Fetched sizes for {len(expected_sizes)} shards.")
except Exception as exc:
    print(f"  ⚠ Could not fetch HF metadata: {exc}")
    print("  Falling back to size-range heuristic only.")

# Compare local sizes against expected
size_matched = []     # Paths that match expected size
size_mismatched = []  # (filename, local_size, expected_size)
size_unknown = []     # Present but no expected size available

for name in sorted(present_names & expected_names):
    local_path = local_shard_map[name]
    local_size = Path(local_path).stat().st_size

    if name in expected_sizes:
        if local_size == expected_sizes[name]:
            size_matched.append(local_path)
        else:
            size_mismatched.append(
                (name, local_size, expected_sizes[name])
            )
    else:
        # No metadata available — accept if size is reasonable
        # (shards are typically 130-175 MB)
        if 100_000_000 < local_size < 200_000_000:
            size_unknown.append(local_path)
        else:
            size_mismatched.append(
                (name, local_size, "unknown (out of range)")
            )

print(f"  Size-matched:    {len(size_matched)}")
print(f"  Size-mismatched: {len(size_mismatched)}")
if size_unknown:
    print(f"  No metadata:     {len(size_unknown)} (accepted by heuristic)")
if size_mismatched:
    print("  ⚠ MISMATCHED FILES (likely truncated):")
    for name, got, want in size_mismatched:
        print(f"    {name}: {got:,} bytes (expected {want:,})")


# ── Layer 3: Tar Structural Integrity ──
print()
print("=" * 60)
print("LAYER 3: Tar Structural Integrity")
print("=" * 60)

# Only check shards that passed the size check
candidates = size_matched + size_unknown
tar_ok = []
tar_corrupted = []  # (path, error_message)

for idx, shard_path in enumerate(candidates):
    try:
        with tarfile.open(shard_path, "r") as tf:
            # Iterate all members to verify the archive index
            _ = tf.getmembers()
        tar_ok.append(shard_path)
    except (tarfile.ReadError, EOFError, OSError) as exc:
        tar_corrupted.append((shard_path, str(exc)))

    # Progress reporting every 100 shards
    if (idx + 1) % 100 == 0 or (idx + 1) == len(candidates):
        print(
            f"  Checked {idx + 1}/{len(candidates)} | "
            f"OK={len(tar_ok)} Corrupt={len(tar_corrupted)}"
        )

if tar_corrupted:
    print("  ⚠ CORRUPTED TAR FILES:")
    for path, err in tar_corrupted:
        print(f"    {Path(path).name}: {err}")


# ── Layer 4: Sample JPEG Decode Spot-Check ──
print()
print("=" * 60)
print("LAYER 4: JPEG Decode Spot-Check")
print("=" * 60)

# Test every SPOT_CHECK_EVERY-th healthy shard
spot_check_shards = tar_ok[::SPOT_CHECK_EVERY]
decode_ok = 0
decode_fail = []  # (path, error_message)

for shard_path in spot_check_shards:
    try:
        with tarfile.open(shard_path, "r") as tf:
            # Find the first JPEG member
            for member in tf.getmembers():
                if member.name.lower().endswith(
                    (".jpg", ".jpeg")
                ):
                    f = tf.extractfile(member)
                    if f is not None:
                        img_data = f.read()
                        img = Image.open(io.BytesIO(img_data))
                        img.verify()  # Validates JPEG structure
                        decode_ok += 1
                        break
    except Exception as exc:
        decode_fail.append((shard_path, str(exc)))

print(
    f"  Spot-checked: {len(spot_check_shards)} shards | "
    f"OK={decode_ok} Failed={len(decode_fail)}"
)
if decode_fail:
    print("  ⚠ DECODE FAILURES:")
    for path, err in decode_fail:
        print(f"    {Path(path).name}: {err}")


# ── Build HEALTHY_SHARDS list ──
# Only shards that passed all applicable layers enter the pipeline
corrupted_set = {p for p, _ in tar_corrupted}
decode_fail_set = {p for p, _ in decode_fail}
HEALTHY_SHARDS = sorted(
    p for p in tar_ok
    if p not in corrupted_set and p not in decode_fail_set
)


# ── Summary Report ──
print()
print("=" * 60)
print("SHARD INTEGRITY REPORT")
print("=" * 60)
print(f"  Total expected:     {EXPECTED_SHARD_COUNT}")
print(f"  Present on disk:    {len(present_names)}")
print(f"  Missing:            {len(missing_names)}")
print(f"  Size-matched:       {len(size_matched)}")
print(f"  Size-mismatched:    {len(size_mismatched)}")
print(f"  Tar-corrupted:      {len(tar_corrupted)}")
print(
    f"  Decode spot-check:  "
    f"{decode_ok}/{len(spot_check_shards)} OK"
)
print(f"  HEALTHY_SHARDS:     {len(HEALTHY_SHARDS)}")
print()

# Final status
issues = len(size_mismatched) + len(tar_corrupted) + len(decode_fail)
if issues == 0 and len(HEALTHY_SHARDS) == EXPECTED_SHARD_COUNT:
    print("Status: ✅ All 1024 shards present and healthy")
elif issues == 0:
    print(
        f"Status: ✅ All {len(HEALTHY_SHARDS)} present shards "
        f"are healthy"
    )
    print(
        f"        ⚠ {len(missing_names)} shards still "
        f"downloading / missing"
    )
else:
    print(
        f"Status: ❌ {issues} issue(s) found — "
        f"only {len(HEALTHY_SHARDS)} shards usable"
    )
    print(
        "        Re-download mismatched/corrupted shards "
        "before running the pipeline."
    )

print("=" * 60)


LAYER 1: Presence Check
  Expected:   1024
  Present:    1024
  Missing:    0

LAYER 2: Size Match (HuggingFace LFS)
  Fetching expected sizes from HuggingFace API...
  ⚠ Could not fetch HF metadata: HTTP Error 400: Bad Request
  Falling back to size-range heuristic only.
  Size-matched:    997
  Size-mismatched: 0
  No metadata:     27 (accepted by heuristic)

LAYER 3: Tar Structural Integrity
  Checked 100/1024 | OK=100 Corrupt=0
  Checked 200/1024 | OK=200 Corrupt=0
  Checked 300/1024 | OK=300 Corrupt=0
  Checked 400/1024 | OK=400 Corrupt=0
  Checked 500/1024 | OK=500 Corrupt=0
  Checked 600/1024 | OK=600 Corrupt=0
  Checked 700/1024 | OK=700 Corrupt=0
  Checked 800/1024 | OK=800 Corrupt=0
  Checked 900/1024 | OK=900 Corrupt=0
  Checked 1000/1024 | OK=1000 Corrupt=0
  Checked 1024/1024 | OK=1024 Corrupt=0

LAYER 4: JPEG Decode Spot-Check
  Spot-checked: 21 shards | OK=21 Failed=0

SHARD INTEGRITY REPORT
  Total expected:     1024
  Present on disk:    1024
  Missing:            0
  

## 4c. DataLoader Smoke Test

Quick sanity check that `make_dataloader()` works with the
verified `HEALTHY_SHARDS` list — pull a single batch and
confirm shapes and keys look correct.


In [6]:
"""Smoke-test the DataLoader using only healthy shards."""

print("Testing DataLoader (1 batch from HEALTHY_SHARDS)...")
test_dl = make_dataloader(batch_size=4)
try:
    test_batch = next(iter(test_dl))
    imgs, keys = test_batch
    print(f"  Got batch: {imgs.shape[0]} images, shape={imgs.shape} (OK)")
    print(f"  Sample keys: {keys[:2]}")
except Exception as e:
    print(f"  FAILED: {type(e).__name__}: {e}")
    raise
finally:
    del test_dl

print("\nDataLoader smoke test: PASSED")


Testing DataLoader (1 batch from HEALTHY_SHARDS)...
  Got batch: 4 images, shape=torch.Size([4, 3, 224, 224]) (OK)
  Sample keys: ['n02097047_2079', 'n01682714_8546']

DataLoader smoke test: PASSED


## 5. Helper Functions

### Image Conversion & Cropping

- **`tensor_to_pil`**: De-normalize and convert a tensor to PIL Image.
- **`crop_from_tensor_and_feat`**: Spatially crop around the maximally
  activating region using the activation map as a guide.

### Scoring

- **`reduce_all_channels`**: Spatial reduction (`max` or `mean`) →
  per-channel scalar score.

### File Safety

- **`_safe_filename`**: Sanitizes strings for cross-platform filenames
  (handles Windows reserved names, special characters, length limits).

In [7]:
"""Image conversion, cropping, scoring, and filename helpers."""

# ImageNet normalization constants
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

# Windows reserved filenames
_WINDOWS_RESERVED = frozenset({
    "CON", "PRN", "AUX", "NUL",
    *(f"COM{i}" for i in range(1, 10)),
    *(f"LPT{i}" for i in range(1, 10)),
})


def tensor_to_pil(x: torch.Tensor) -> Image.Image:
    """De-normalize a tensor and convert to PIL Image.

    Args:
        x: Tensor of shape (C, H, W) or (1, C, H, W).

    Returns:
        PIL Image in RGB mode.
    """
    x = x.detach().cpu()
    if x.dim() == 4:
        x = x[0]
    x = (x * STD + MEAN).clamp(0, 1)
    x = (x.permute(1, 2, 0).numpy() * 255).astype(np.uint8)
    return Image.fromarray(x)


def crop_from_tensor_and_feat(
    img_tensor_norm: torch.Tensor,
    feat_single: torch.Tensor,
    channel_id: int,
    frac: float = 0.25,
) -> Image.Image:
    """Crop the image around the maximally activating spatial location.

    Args:
        img_tensor_norm: Normalized image tensor (C, H, W).
        feat_single: Activation tensor for one sample (C, h, w).
        channel_id: Which channel to use for locating the max activation.
        frac: Fraction of the image size for the crop window.

    Returns:
        Cropped PIL Image centered on the max activation.
    """
    img = (img_tensor_norm.detach().cpu() * STD + MEAN).clamp(0, 1)
    img_np = (img.permute(1, 2, 0).numpy() * 255).astype(np.uint8)
    h_img, w_img = img_np.shape[:2]

    act = feat_single[channel_id].detach().cpu().clamp(min=0)
    h_act, w_act = act.shape

    flat_idx = int(torch.argmax(act).item())
    iy, ix = flat_idx // w_act, flat_idx % w_act

    cy = int((iy + 0.5) * h_img / h_act)
    cx = int((ix + 0.5) * w_img / w_act)

    ch, cw = int(h_img * frac), int(w_img * frac)
    y0 = max(0, cy - ch // 2)
    y1 = min(h_img, y0 + ch)
    x0 = max(0, cx - cw // 2)
    x1 = min(w_img, x0 + cw)

    return Image.fromarray(img_np[y0:y1, x0:x1])


@torch.no_grad()
def reduce_all_channels(
    feat: torch.Tensor,
    reduction: str,
) -> torch.Tensor:
    """Reduce spatial dims of feature map to per-channel scalars.

    Args:
        feat: Feature tensor of shape (B, C, h, w).
        reduction: Either 'max' or 'mean'.

    Returns:
        Tensor of shape (B, C).

    Raises:
        ValueError: If reduction is not 'max' or 'mean'.
    """
    if reduction == "max":
        return feat.amax(dim=(2, 3))
    elif reduction == "mean":
        return feat.mean(dim=(2, 3))
    else:
        raise ValueError(
            f"reduction must be 'max' or 'mean', got '{reduction}'"
        )


def _safe_filename(s: str, max_len: int = 120) -> str:
    """Sanitize a string for use as a filename on any OS.

    Handles path separators, special characters, Windows reserved
    names (CON, NUL, etc.), and length limits.

    Args:
        s: Raw string to sanitize.
        max_len: Maximum filename length.

    Returns:
        Sanitized filename string.
    """
    s = str(s).replace("/", "_").replace("\\", "_")
    s = re.sub(r"[^a-zA-Z0-9._-]+", "_", s)
    s = s[:max_len]
    name_part = s.split(".")[0].upper()
    if name_part in _WINDOWS_RESERVED:
        s = f"_{s}"
    return s


print("Helpers loaded.")

Helpers loaded.


## 6. Checkpoint System

Atomic writes with a temporary file ensure that a crash during save
never corrupts the checkpoint. On load, corrupt files are detected
and the scan restarts cleanly.

```
save_ckpt: data → .tmp file → os.replace → .pkl (atomic)
load_ckpt: .pkl → validate keys → return dict | None
```

In [8]:
"""Checkpoint save/load with atomic writes and corruption recovery."""

CKPT_PATH = CFG["CKPT_DIR"] / "topk_scan_ckpt.pkl"
PASS2_CKPT_PATH = CFG["CKPT_DIR"] / "pass2_save_ckpt.pkl"


def save_ckpt(
    path: Path,
    step: int,
    heaps: dict,
    ch0: int,
    ch1: int,
) -> None:
    """Save checkpoint atomically (write-then-rename).

    Writes to a temporary file first, then performs an atomic rename
    so a crash mid-write never corrupts the checkpoint.

    Args:
        path: Checkpoint file path.
        step: Current batch step index.
        heaps: Dictionary mapping channel_id -> min-heap list.
        ch0: Start of channel range.
        ch1: End of channel range (inclusive).
    """
    tmp_path = path.with_suffix(".tmp")
    with open(tmp_path, "wb") as f:
        pkl.dump(
            {"step": step, "heaps": heaps, "ch0": ch0, "ch1": ch1},
            f,
            protocol=pkl.HIGHEST_PROTOCOL,
        )
    os.replace(tmp_path, path)  # Atomic on all platforms


def load_ckpt(path: Path) -> dict | None:
    """Load checkpoint, returning None if missing or corrupt.

    Args:
        path: Checkpoint file path.

    Returns:
        Checkpoint dict, or None if missing/corrupt.
    """
    if not path.exists():
        return None
    try:
        with open(path, "rb") as f:
            data = pkl.load(f)
        return data
    except (pkl.UnpicklingError, EOFError, ValueError, KeyError) as exc:
        print(f"Warning: Corrupt checkpoint ({exc}), starting fresh.")
        return None


def save_pass2_ckpt(path: Path, saved_keys: set) -> None:
    """Save Pass 2 checkpoint (set of already-saved image keys).

    Args:
        path: Checkpoint file path.
        saved_keys: Set of WDS keys that have been saved to disk.
    """
    tmp_path = path.with_suffix(".tmp")
    with open(tmp_path, "wb") as f:
        pkl.dump({"saved_keys": saved_keys}, f, protocol=pkl.HIGHEST_PROTOCOL)
    os.replace(tmp_path, path)


def load_pass2_ckpt(path: Path) -> set:
    """Load Pass 2 checkpoint, returning empty set if missing/corrupt.

    Args:
        path: Checkpoint file path.

    Returns:
        Set of previously saved image keys.
    """
    if not path.exists():
        return set()
    try:
        with open(path, "rb") as f:
            data = pkl.load(f)
        keys = data.get("saved_keys", set())
        if isinstance(keys, set):
            return keys
        return set()
    except (pkl.UnpicklingError, EOFError, ValueError, KeyError) as exc:
        print(f"Warning: Corrupt Pass 2 checkpoint ({exc}), starting fresh.")
        return set()


print(f"Pass 1 checkpoint: {CKPT_PATH} (exists={CKPT_PATH.exists()})")
print(f"Pass 2 checkpoint: {PASS2_CKPT_PATH} (exists={PASS2_CKPT_PATH.exists()})")


Pass 1 checkpoint: c:\Users\cataluna84\Documents\Workspace\VisionInterpretability\notebooks\results\checkpoints\topk_scan_ckpt.pkl (exists=False)
Pass 2 checkpoint: c:\Users\cataluna84\Documents\Workspace\VisionInterpretability\notebooks\results\checkpoints\pass2_save_ckpt.pkl (exists=False)


## 7. Pass 1 — Top-K Activation Scan

For each channel in `[CHANNEL_START, CHANNEL_END]`, maintain a
**min-heap** of size `TOPK` keyed by the spatial-max activation score:

$$\text{score}_{c}(x) = \max_{i,j} \; A^c_{i,j}(x)$$

### Memory Management

- **AMP (FP16)** halves activation memory during inference.
- **Explicit `del feat`** releases the activation tensor each iteration.
- **`torch.cuda.empty_cache()`** called at each checkpoint save.

### Checkpoint Resume

If a checkpoint exists, the scan resumes from the saved batch step.
The channel range is validated against the checkpoint metadata.

In [9]:
"""Pass 1: Scan the dataset to find top-k keys per channel."""


def scan_topk_keys(
    model: torch.nn.Module,
    device: torch.device,
    layer_name: str,
    channel_start: int,
    channel_end_inclusive: int,
    topk: int,
    heap_batch: int,
    reduction: str,
    batch_size: int,
    save_every: int,
) -> dict:
    """Scan the full dataset and return top-k keys per channel.

    Uses a min-heap per channel to efficiently track the top-k
    highest-scoring image keys across the entire dataset.

    Args:
        model: Pretrained InceptionV1 model in eval mode.
        device: Torch device (cuda or cpu).
        layer_name: Name of the target layer.
        channel_start: First channel index (inclusive).
        channel_end_inclusive: Last channel index (inclusive).
        topk: Number of top-scoring images to track per channel.
        heap_batch: Number of top scores to extract per batch.
        reduction: Spatial reduction method ('max' or 'mean').
        batch_size: DataLoader batch size.
        save_every: Checkpoint save frequency (in batches).

    Returns:
        Dict mapping channel_id → list of (score, key) tuples,
        sorted descending by score.
    """
    activation = {}

    def hook_fn(module, inp, out):
        """Forward hook to capture layer activations."""
        activation["feat"] = out

    layer = dict(model.named_modules())[layer_name]
    handle = layer.register_forward_hook(hook_fn)

    try:
        model.eval()
        heaps = None
        ch0 = None
        ch1 = None
        start_step = 0
        last_step = None

        # Load checkpoint if exists
        ckpt = load_ckpt(CKPT_PATH)
        if ckpt is not None:
            start_step = int(ckpt["step"]) + 1
            heaps = ckpt["heaps"]
            ch0 = int(ckpt["ch0"])
            ch1 = int(ckpt["ch1"])
            print(f"✓ Resuming from batch {start_step} (channels [{ch0}, {ch1}])")

        # Create fresh DataLoader (WDS is single-use)
        dl_train = make_dataloader(batch_size=batch_size)

        with torch.no_grad():
            for step, (imgs, keys) in enumerate(dl_train):
                # Skip batches already processed (checkpoint resume)
                if step < start_step:
                    continue

                last_step = step
                imgs = imgs.to(device, non_blocking=True)

                # Forward pass with AMP for memory efficiency
                use_cuda = imgs.is_cuda
                with torch.amp.autocast(
                    device_type="cuda", enabled=use_cuda,
                ):
                    _ = model(imgs)

                feat = activation["feat"]  # [B, C, h, w]
                B, C = feat.shape[0], feat.shape[1]

                # Initialize heaps on first batch (after seeing C)
                if heaps is None:
                    ch0 = max(0, int(channel_start))
                    ch1 = min(C - 1, int(channel_end_inclusive))
                    if ch0 > ch1:
                        raise ValueError(
                            f"Invalid channel range after clamping: "
                            f"[{ch0}, {ch1}] with C={C}"
                        )
                    heaps = {c: [] for c in range(ch0, ch1 + 1)}
                    print(
                        f"Scanning layer={layer_name} channels={C} "
                        f"using range [{ch0}, {ch1}]"
                    )
                else:
                    # Validate checkpoint channel range
                    ch0 = max(0, min(ch0, C - 1))
                    ch1 = max(0, min(ch1, C - 1))
                    if ch0 > ch1:
                        raise ValueError(
                            f"Checkpoint channel range invalid: "
                            f"[{ch0}, {ch1}] with C={C}"
                        )

                # Compute per-channel scores
                scores_bc = reduce_all_channels(feat, reduction).float()
                scores_sel = scores_bc[:, ch0:ch1 + 1]  # [B, Cr]

                k = min(heap_batch, B)
                vals, idxs = torch.topk(scores_sel, k=k, dim=0)

                vals = vals.detach().cpu()
                idxs = idxs.detach().cpu()

                # Update min-heaps
                for offset, c in enumerate(range(ch0, ch1 + 1)):
                    h = heaps[c]
                    for j in range(k):
                        v = float(vals[j, offset].item())
                        i = int(idxs[j, offset].item())
                        key = keys[i]
                        item = (v, key)
                        if len(h) < topk:
                            heapq.heappush(h, item)
                        elif v > h[0][0]:
                            heapq.heapreplace(h, item)

                # Free activation memory
                del feat, scores_bc, scores_sel, vals, idxs

                # Progress reporting
                if (step + 1) % 50 == 0:
                    sample_c = ch0
                    if heaps[sample_c]:
                        min_score = heaps[sample_c][0][0]
                        print(
                            f"  batch {step + 1:>6d} | "
                            f"ch{sample_c} min_top{topk}={min_score:.4f}"
                        )
                    if device.type == "cuda":
                        mem_mb = torch.cuda.memory_allocated(0) / 1024**2
                        print(f"             | GPU mem: {mem_mb:.0f} MB")

                # Periodic checkpoint
                if (step + 1) % save_every == 0:
                    save_ckpt(CKPT_PATH, step, heaps, ch0, ch1)
                    print(f"  ✓ Checkpoint saved at batch {step + 1}")
                    if device.type == "cuda":
                        torch.cuda.empty_cache()

        # Final checkpoint
        if heaps is not None and last_step is not None:
            save_ckpt(CKPT_PATH, last_step, heaps, ch0, ch1)
            print(f"  ✓ Final checkpoint saved at batch {last_step + 1}")

        # Return sorted results
        if heaps is None:
            print("ERROR: No batches processed. Check HF_TOKEN and network.")
            return {}

        topk_by_channel = {
            c: sorted(h, key=lambda x: x[0], reverse=True)
            for c, h in heaps.items()
        }
        return topk_by_channel

    finally:
        handle.remove()
        print("Hook removed.")

## 8. Pass 2 -- Save Top-K Images (with Resume)

Re-stream the dataset, match keys found in Pass 1, and save:

- **FULL**: Full 224x224 de-normalized image.
- **CROP**: Spatially cropped patch around max-activation region.

### Checkpoint/Resume

Pass 2 saves its own checkpoint tracking which keys have been
written to disk. If interrupted, re-running will skip already-saved
images automatically.

### Edge Cases Handled

| Edge Case | Mitigation |
|-----------|------------|
| Interrupted mid-save | Checkpoint tracks saved keys, resume skips them |
| Per-image save failure | `try/except` per image, log & continue |
| WDS DataLoader exhausted | Fresh `make_dataloader()` call |
| Hook not removed on error | `try/finally` block |
| All keys found early | Early exit + final checkpoint save |


In [10]:
"""Pass 2: Re-stream dataset and save top-k images to disk (with resume)."""


def map_top_keys(
    topk_by_channel: dict,
) -> tuple[set, dict]:
    """Build lookup structures from the top-k results.

    Args:
        topk_by_channel: Dict mapping channel_id -> sorted list of
            (score, key) tuples.

    Returns:
        Tuple of:
        - wanted_keys: Set of all unique image keys to find.
        - key_to_targets: Dict mapping key -> list of
          (channel, score, rank) tuples.
    """
    wanted_keys = set()
    key_to_targets = {}  # key -> list[(channel, score, rank)]

    for c, items in topk_by_channel.items():
        for rank, (score, key) in enumerate(items, start=1):
            wanted_keys.add(key)
            key_to_targets.setdefault(key, []).append(
                (c, float(score), rank)
            )

    return wanted_keys, key_to_targets


def save_topk_images(
    model: torch.nn.Module,
    device: torch.device,
    layer_name: str,
    topk_by_channel: dict,
    out_root: Path,
    crop_frac: float,
    batch_size: int,
    save_every: int,
) -> None:
    """Re-stream dataset and save top-k images to disk.

    Supports checkpoint/resume: if interrupted, re-running will skip
    images that were already saved in a previous run.

    Args:
        model: Pretrained InceptionV1 model in eval mode.
        device: Torch device.
        layer_name: Target layer name.
        topk_by_channel: Results from scan_topk_keys().
        out_root: Root directory for saved images.
        crop_frac: Crop fraction for spatial cropping.
        batch_size: DataLoader batch size.
        save_every: Checkpoint save frequency (in batches).
    """
    # Build lookup maps
    wanted_keys, key_to_targets = map_top_keys(topk_by_channel)
    total_keys = len(wanted_keys)

    # Load Pass 2 checkpoint (resume support)
    previously_saved = load_pass2_ckpt(PASS2_CKPT_PATH)
    if previously_saved:
        print(f"Resuming Pass 2: {len(previously_saved)} keys already saved.")

    # Subtract already-saved keys
    remaining = wanted_keys - previously_saved
    saved_keys = set(previously_saved)  # mutable copy for tracking
    print(f"Looking for {len(remaining)} keys ({total_keys} total, "
          f"{len(saved_keys)} already saved).")

    if not remaining:
        print("All keys already saved! Nothing to do.")
        return

    # Hook setup
    activation = {}

    def hook_fn(module, inp, out):
        """Forward hook to capture layer activations."""
        activation["feat"] = out

    layer = dict(model.named_modules())[layer_name]
    handle = layer.register_forward_hook(hook_fn)

    layer_dir = out_root / layer_name
    layer_dir.mkdir(parents=True, exist_ok=True)

    try:
        model.eval()

        # Fresh DataLoader (WDS iterators are single-use)
        dl_train = make_dataloader(batch_size=batch_size)

        for step, (imgs, keys) in enumerate(dl_train):
            imgs = imgs.to(device, non_blocking=True)

            use_cuda = imgs.is_cuda
            with torch.amp.autocast(
                device_type="cuda", enabled=use_cuda,
            ):
                _ = model(imgs)

            feat = activation["feat"]  # [B, C, h, w]

            # Find which indices in this batch we need
            hit_indices = [
                i for i, k in enumerate(keys) if k in remaining
            ]

            if not hit_indices:
                del feat
                if (step + 1) % 200 == 0:
                    print(
                        f"  batch {step + 1:>6d} | "
                        f"remaining={len(remaining)}/{total_keys}"
                    )
                continue

            for i in hit_indices:
                key = keys[i]
                key_safe = _safe_filename(key)

                # Full-size de-normalized image (computed once per sample)
                pil_full = tensor_to_pil(imgs[i])

                # Save for each channel that selected this key
                for (c, score, rank) in key_to_targets[key]:
                    ch_dir = layer_dir / f"ch_{c:04d}"
                    ch_dir.mkdir(parents=True, exist_ok=True)

                    try:
                        pil_crop = crop_from_tensor_and_feat(
                            imgs[i], feat[i], c, frac=crop_frac,
                        )
                        full_path = (
                            ch_dir
                            / f"rank{rank:02d}_FULL_"
                              f"score{score:.4f}_{key_safe}.jpg"
                        )
                        crop_path = (
                            ch_dir
                            / f"rank{rank:02d}_CROP_"
                              f"score{score:.4f}_{key_safe}.jpg"
                        )
                        pil_full.save(full_path)
                        pil_crop.save(crop_path)
                    except (OSError, IOError) as exc:
                        print(
                            f"  Warning: Failed to save ch{c}/rank{rank} "
                            f"key={key_safe}: {exc}"
                        )
                        continue

                remaining.discard(key)
                saved_keys.add(key)

            # Free activation memory
            del feat

            # Progress report
            if (step + 1) % 50 == 0:
                num_saved = total_keys - len(remaining)
                print(
                    f"  batch {step + 1:>6d} | "
                    f"saved={num_saved}/{total_keys} "
                    f"remaining={len(remaining)}"
                )

            # Periodic checkpoint save
            if (step + 1) % save_every == 0:
                save_pass2_ckpt(PASS2_CKPT_PATH, saved_keys)
                print(f"  Checkpoint saved at batch {step + 1} "
                      f"({len(saved_keys)} keys saved)")

            # Early exit when all keys found
            if not remaining:
                print(f"  All keys found! Done at batch {step + 1}.")
                break

    finally:
        handle.remove()

        # Always save final checkpoint
        save_pass2_ckpt(PASS2_CKPT_PATH, saved_keys)
        print(f"Hook removed. Final checkpoint saved "
              f"({len(saved_keys)} keys).")
        print(f"Output -> {layer_dir}")

    if remaining:
        print(
            f"Warning: {len(remaining)} keys not found in dataset "
            f"(possible streaming/shard mismatch)."
        )


## 9. Run the Pipeline

Execute both passes sequentially with GPU memory cleanup in between.

- **Pass 1** streams ~1.28M images to build top-k heaps.
- **Pass 2** re-streams to find and save the selected images.
- Interrupt and re-run at any time — checkpoints handle resume.

In [11]:
"""Execute the full two-pass pipeline."""

print("=" * 60)
print("PASS 1: Scanning top-k activations")
print("=" * 60)

topk_by_channel = scan_topk_keys(
    model=model,
    device=device,
    layer_name=CFG["LAYER_NAME"],
    channel_start=CFG["CHANNEL_START"],
    channel_end_inclusive=CFG["CHANNEL_END"],
    topk=CFG["TOPK"],
    heap_batch=CFG["HEAP_BATCH"],
    reduction=CFG["REDUCTION"],
    batch_size=CFG["BATCH_SIZE"],
    save_every=CFG["SAVE_EVERY"],
)

# Free GPU memory between passes
gc.collect()
if device.type == "cuda":
    torch.cuda.empty_cache()
    free_mem = torch.cuda.mem_get_info(0)[0] / 1024**3
    print(f"GPU free after Pass 1 cleanup: {free_mem:.1f} GB")

print()
print("=" * 60)
print("PASS 2: Saving images to disk")
print("=" * 60)

save_topk_images(
    model=model,
    device=device,
    layer_name=CFG["LAYER_NAME"],
    topk_by_channel=topk_by_channel,
    out_root=CFG["OUT_ROOT"],
    crop_frac=CFG["CROP_FRAC"],
    batch_size=CFG["BATCH_SIZE"],
    save_every=CFG["SAVE_EVERY"],
)

print()
print("=" * 60)
print("Pipeline complete!")
print(f"  Results saved to: {CFG['OUT_ROOT']}")
print("=" * 60)


PASS 1: Scanning top-k activations
Scanning layer=mixed5a channels=832 using range [0, 831]
  batch     50 | ch0 min_top10=177.3750
             | GPU mem: 161 MB
  batch    100 | ch0 min_top10=198.5000
             | GPU mem: 161 MB
  batch    150 | ch0 min_top10=200.7500
             | GPU mem: 161 MB
  batch    200 | ch0 min_top10=202.8750
             | GPU mem: 161 MB
  batch    250 | ch0 min_top10=211.0000
             | GPU mem: 161 MB
  batch    300 | ch0 min_top10=216.0000
             | GPU mem: 161 MB
  batch    350 | ch0 min_top10=223.3750
             | GPU mem: 161 MB
  batch    400 | ch0 min_top10=227.0000
             | GPU mem: 161 MB
  batch    450 | ch0 min_top10=233.6250
             | GPU mem: 161 MB
  batch    500 | ch0 min_top10=235.2500
             | GPU mem: 161 MB
  ✓ Checkpoint saved at batch 500
  batch    550 | ch0 min_top10=236.1250
             | GPU mem: 161 MB
  batch    600 | ch0 min_top10=238.0000
             | GPU mem: 161 MB
  batch    650 | ch0 m